Sentence json -> sentence csv

Builds **sentence-level revision groups** from the paragraph revision groups:

- reads every article from `data/sentence/sentence-json` (versions already split into sentences by spaCy)
- for every paragraph step `v(k-1) -> v(k)`, aligns the two sentence lists:
  - `equal` sentences are unchanged (same text ignoring case / spacing, like data-preprocessing stage 3)
  - `replace` blocks are edits: sentences are paired by **highest BLEU**, every sentence used at most once
    (1 -> 1 blocks are simply that pair; m -> n blocks = splits / merges / several sentences rewritten)
  - `insert` / `delete` and unpaired sentences are additions / removals, not edits
- chains the edits: if an edit's base sentence is the latest version of an existing sentence group, the edit
  continues that group, otherwise it starts a new one (depth 0 = base sentence, depth 1 = edited sentence)
- `prior` / `post` = the rest of the paragraph before / after the sentence, from that row's own version
- **no BLEU filter here**: every pair is kept and `bleu_to_previous` is recorded, the threshold is applied later
  when pairs are built (a weak link must split a group, not just drop a row)

Writes one csv per article to `data/sentence/raw-csv` (same file names as `data/paragraph/raw-csv`) and a summary
to `data/sentence/sentence-csv-summary.csv`. Always rebuilds everything (takes a few minutes), so the summary is complete.

In [ ]:
from pathlib import Path
import csv
import json
import re
from collections import Counter
from difflib import SequenceMatcher

from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu

sentence_json_path = Path("../../data/sentence/sentence-json")
sentence_csv_path = Path("../../data/sentence/raw-csv")
summary_csv_file = Path("../../data/sentence/sentence-csv-summary.csv")

sentence_csv_path.mkdir(parents=True, exist_ok=True)
summary_csv_file.parent.mkdir(parents=True, exist_ok=True)

COLUMNS = ["revision_group_no", "revision_depth", "paragraph_group_no", "paragraph_depth",
           "prior", "text", "post", "bleu_to_previous", "diff_url"]  # diff_url last, like raw-csv

In [ ]:
# "is it the same sentence?" - same rule as data-preprocessing stage 3 (pipeline/utils/text.py norm_key):
# fix scrape spacing around punctuation, collapse whitespace, ignore case
_WHITESPACE_RE = re.compile(r"\s+")
_SPACE_BEFORE_CLOSE_RE = re.compile(r"\s+([.,;:!?)\]}”\"'’])")
_SPACE_AFTER_OPEN_RE = re.compile(r"([(\[{“\"'‘])\s+")
_SPACE_AFTER_PUNCT_RE = re.compile(r"([.,;:!?])\s*")


def norm_key(text):
    t = _WHITESPACE_RE.sub(" ", str(text or "")).strip()
    if not t:
        return ""
    t = _SPACE_BEFORE_CLOSE_RE.sub(r"\1", t)
    t = _SPACE_AFTER_OPEN_RE.sub(r"\1", t)
    t = _WHITESPACE_RE.sub(" ", t).strip()
    t = _SPACE_AFTER_PUNCT_RE.sub(r"\1 ", t)
    t = _WHITESPACE_RE.sub(" ", t).strip()
    return t.casefold()


# same BLEU as data-preprocessing stage 5 (pipeline/utils/bleu.py): case-insensitive, NLTK sentence_bleu, method1
_SMOOTH = SmoothingFunction().method1


def bleu(base, revised):
    reference = base.lower().split()
    hypothesis = revised.lower().split()
    if not reference or not hypothesis:
        return 0.0
    return sentence_bleu([reference], hypothesis, smoothing_function=_SMOOTH)


def pair_block(base_sentences, revised_sentences):
    """Best-BLEU pairing inside one changed block, every sentence used at most once.
    Returns [(base_offset, revised_offset, bleu)] sorted by base_offset."""
    candidates = sorted(
        ((bleu(b, r), bi, ri) for bi, b in enumerate(base_sentences) for ri, r in enumerate(revised_sentences)),
        key=lambda c: (-c[0], c[1], c[2]),
    )
    used_base, used_revised, pairs = set(), set(), []
    for score, bi, ri in candidates:
        if bi in used_base or ri in used_revised:
            continue
        used_base.add(bi)
        used_revised.add(ri)
        pairs.append((bi, ri, score))
    return sorted(pairs)

In [ ]:
def make_row(sentences, index, paragraph_group_no, paragraph_depth, bleu_to_previous, diff_url):
    return {
        "paragraph_group_no": paragraph_group_no,
        "paragraph_depth": paragraph_depth,
        "prior": " ".join(sentences[:index]),
        "text": sentences[index],
        "post": " ".join(sentences[index + 1:]),
        "bleu_to_previous": "" if bleu_to_previous is None else round(bleu_to_previous, 4),
        "diff_url": diff_url or "",
    }


def build_sentence_groups(article, stats, bleu_scores):
    """Return csv rows (sentence revision groups, stacked by depth) for one article."""
    chains = []  # one list of rows per sentence revision group, index in list = revision_depth
    for paragraph_group_no, group in enumerate(article["groups"], start=1):
        versions = [version["sentences"] for version in group["versions"]]
        diff_urls = group.get("diff_urls") or []
        tip = {}  # sentence index in the current version -> chain it is the latest version of
        for k in range(1, len(versions)):
            before, after = versions[k - 1], versions[k]
            stats["paragraph_steps"] += 1
            matcher = SequenceMatcher(a=[norm_key(s) for s in before], b=[norm_key(s) for s in after], autojunk=False)
            new_tip = {}
            for tag, i1, i2, j1, j2 in matcher.get_opcodes():
                if tag == "equal":  # unchanged sentences carry their chain to the new position
                    for offset in range(i2 - i1):
                        if i1 + offset in tip:
                            new_tip[j1 + offset] = tip[i1 + offset]
                elif tag == "delete":
                    stats["sentences_removed"] += i2 - i1
                elif tag == "insert":
                    stats["sentences_added"] += j2 - j1
                else:  # replace = edited sentences
                    pairs = pair_block(before[i1:i2], after[j1:j2])
                    kind = "pairs_1_to_1" if (i2 - i1, j2 - j1) == (1, 1) else "pairs_m_to_n"
                    stats[kind] += len(pairs)
                    stats["sentences_removed"] += (i2 - i1) - len(pairs)
                    stats["sentences_added"] += (j2 - j1) - len(pairs)
                    for base_offset, revised_offset, _pair_score in pairs:
                        i, j = i1 + base_offset, j1 + revised_offset
                        if i in tip:  # base sentence is the latest version of an existing group -> continue it
                            chain = tip[i]
                        else:  # new sentence revision group, depth 0 = base sentence
                            chain = len(chains)
                            chains.append([make_row(before, i, paragraph_group_no, k - 1, None, None)])
                        # BLEU against the previous row of the group (can differ from the pairing score when the
                        # sentence had a case / spacing-only change in between, which does not add a row)
                        score = bleu(chains[chain][-1]["text"], after[j])
                        diff_url = diff_urls[k] if k < len(diff_urls) else None
                        chains[chain].append(make_row(after, j, paragraph_group_no, k, score, diff_url))
                        new_tip[j] = chain
                        bleu_scores.append(score)
            tip = new_tip  # deleted / unpaired sentences end their chain here

    rows = []
    for revision_group_no, chain in enumerate(chains, start=1):
        for revision_depth, row in enumerate(chain):
            rows.append({"revision_group_no": revision_group_no, "revision_depth": revision_depth, **row})
    return rows, [len(chain) - 1 for chain in chains]

In [ ]:
stats = Counter()
depths = Counter()
bleu_scores = []
n_articles = 0

for json_file in sorted(sentence_json_path.glob("*.json")):
    with json_file.open(encoding="utf-8") as jf:
        article = json.load(jf)
    rows, chain_depths = build_sentence_groups(article, stats, bleu_scores)
    depths.update(chain_depths)

    # same file name as data/paragraph/raw-csv, so paragraph and sentence csv line up
    csv_file = sentence_csv_path / f"{article['article_title']}.csv".replace("/", "_")
    # write to a temp file and rename at the end, so a half written file is never left behind
    tmp_csv_file = csv_file.with_suffix(".csv.tmp")
    with tmp_csv_file.open("w", encoding="utf-8", newline="") as cf:
        csv_writer = csv.DictWriter(cf, fieldnames=COLUMNS)
        csv_writer.writeheader()
        csv_writer.writerows(rows)
    tmp_csv_file.rename(csv_file)
    n_articles += 1

n_pairs = stats["pairs_1_to_1"] + stats["pairs_m_to_n"]
summary = [
    ("articles", n_articles),
    ("paragraph_steps", stats["paragraph_steps"]),
    ("sentence_pairs", n_pairs),
    ("sentence_pairs_1_to_1", stats["pairs_1_to_1"]),
    ("sentence_pairs_m_to_n", stats["pairs_m_to_n"]),
    ("sentences_added_not_paired", stats["sentences_added"]),
    ("sentences_removed_not_paired", stats["sentences_removed"]),
    ("sentence_revision_groups", sum(depths.values())),
]
summary += [(f"groups_depth_{d}", depths[d]) for d in range(1, 6)]
summary += [("groups_depth_6_or_more", sum(n for d, n in depths.items() if d >= 6))]
summary += [(f"pairs_bleu_above_{t}", sum(s > t for s in bleu_scores)) for t in (0, 0.1, 0.2, 0.3)]

with summary_csv_file.open("w", encoding="utf-8", newline="") as sf:
    summary_writer = csv.writer(sf)
    summary_writer.writerow(["metric", "value"])
    summary_writer.writerows(summary)

for name, value in summary:
    print(f"{name:32} {value}")

In [ ]:
# TEST SENTENCE CSV

n_json = len(list(sentence_json_path.glob("*.json")))
n_csv = len(list(sentence_csv_path.glob("*.csv")))
n_tmp = len(list(sentence_csv_path.glob("*.tmp")))
print("sentence-json files:", n_json, " sentence csv files:", n_csv, " left over tmp:", n_tmp)
assert n_json == n_csv and n_tmp == 0

# example: the article title sentence is edited at several steps, and another sentence in between
with (sentence_csv_path / "10-Signs-of-Trauma-Bonding.csv").open(encoding="utf-8") as cf:
    for row in list(csv.DictReader(cf))[:12]:
        print(row["revision_group_no"], row["revision_depth"], f"(paragraph step {row['paragraph_depth']})",
              row["bleu_to_previous"] or "-", "|", row["text"][:80])